<a href="https://colab.research.google.com/github/vamsi54-bit/AI-Projects/blob/main/leaflens/notebooks/01_train_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/leaflens/datasets

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [ ]:
!pip install -q kaggle

In [ ]:
!kaggle datasets list -s plantvillage | head -5

ref                                                       title                                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------  -------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
abdallahalidev/plantvillage-dataset                       PlantVillage Dataset                         4371949460  2019-09-01 11:52:26.883000         130567       1125  0.875            
emmarex/plantdisease                                      PlantVillage Dataset                          689512690  2018-10-30 01:16:23.440000         182834       1288  0.5625           


In [ ]:
%cd /content/drive/MyDrive/leaflens/datasets
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d plantvillage
!rm plantdisease.zip

/content/drive/MyDrive/leaflens/datasets
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
plantdisease.zip: Skipping, found more recently modified local copy (use --force to force download)
replace plantvillage/PlantVillage/Pepper__bell___Bacterial_spot/0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Spot 8964.JPG? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
%cd /content/drive/MyDrive/leaflens/datasets
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git plantdoc
!ls plantdoc

/content/drive/MyDrive/leaflens/datasets
Cloning into 'plantdoc'...
remote: Enumerating objects: 2628, done.
remote: Counting objects: 100% (2628/2628), done.
remote: Compressing objects: 100% (2627/2627), done.
remote: Total 2628 (delta 1), reused 2616 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (2628/2628), 932.91 MiB | 16.64 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
!rm -rf /content/data && mkdir -p /content/data
%cd /content/data

# PlantVillage: download and unzip locally
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d plantvillage

# back up just the zip to Drive (one big file is safe there)
!cp plantdisease.zip /content/drive/MyDrive/leaflens/datasets/

# PlantDoc: clone locally
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git plantdoc

!ls /content/data/plantvillage
!ls /content/data/plantdoc

/content/data
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:41<00:00, 16.5MB/s]

Cloning into 'plantdoc'...
remote: Enumerating objects: 2628, done.
remote: Counting objects: 100% (2628/2628), done.
remote: Compressing objects: 100% (2627/2627), done.
remote: Total 2628 (delta 1), reused 2616 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (2628/2628), 932.91 MiB | 17.28 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (2581/2581), done.
plantvillage  PlantVillage
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
import os

base = "/content/data"

pv = f"{base}/plantvillage/PlantVillage"
pv_classes = sorted(os.listdir(pv))
print("PlantVillage classes:", len(pv_classes))
for c in pv_classes:
    print(f"  {c}: {len(os.listdir(f'{pv}/{c}'))} images")

for split in ["train", "test"]:
    p = f"{base}/plantdoc/{split}"
    classes = sorted(os.listdir(p))
    total = sum(len(os.listdir(f'{p}/{c}')) for c in classes)
    print(f"\nPlantDoc {split}: {len(classes)} classes, {total} images")

PlantVillage classes: 15
  Pepper__bell___Bacterial_spot: 997 images
  Pepper__bell___healthy: 1478 images
  Potato___Early_blight: 1000 images
  Potato___Late_blight: 1000 images
  Potato___healthy: 152 images
  Tomato_Bacterial_spot: 2127 images
  Tomato_Early_blight: 1000 images
  Tomato_Late_blight: 1909 images
  Tomato_Leaf_Mold: 952 images
  Tomato_Septoria_leaf_spot: 1771 images
  Tomato_Spider_mites_Two_spotted_spider_mite: 1676 images
  Tomato__Target_Spot: 1404 images
  Tomato__Tomato_YellowLeaf__Curl_Virus: 3209 images
  Tomato__Tomato_mosaic_virus: 373 images
  Tomato_healthy: 1591 images

PlantDoc train: 28 classes, 2342 images

PlantDoc test: 27 classes, 236 images


In [ ]:
!pip install -q timm

import os, random, torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

PV = "/content/data/plantvillage/PlantVillage"

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# two views of the same folder so train gets augmentation, val doesn't
full_train = datasets.ImageFolder(PV, transform=train_tf)
full_val = datasets.ImageFolder(PV, transform=val_tf)
classes = full_train.classes
print("Classes:", len(classes))

# 80/20 split with a fixed seed
g = torch.Generator().manual_seed(42)
n = len(full_train)
idx = torch.randperm(n, generator=g).tolist()
split = int(0.8 * n)
train_idx, val_idx = idx[:split], idx[split:]

train_ds = torch.utils.data.Subset(full_train, train_idx)
val_ds = torch.utils.data.Subset(full_val, val_idx)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("Train:", len(train_ds), "Val:", len(val_ds))

Device: cuda
Classes: 15
Train: 16510 Val: 4128


In [ ]:
import timm, numpy as np, torch.nn as nn
from sklearn.metrics import f1_score

# EfficientNet-B0 pretrained, with a new 15-class head
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(classes)).to(device)

# class weights to handle the imbalance (rare classes count more)
targets = np.array(full_train.targets)[train_idx]
counts = np.bincount(targets, minlength=len(classes))
weights = torch.tensor(counts.sum() / (len(classes) * counts), dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
EPOCHS = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, steps_per_epoch=len(train_dl), epochs=EPOCHS
)
scaler = torch.cuda.amp.GradScaler()

SAVE = "/content/drive/MyDrive/leaflens/effnet_b0_pv.pt"
best_acc = 0

for epoch in range(EPOCHS):
    # train
    model.train()
    total_loss = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    # validate
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in val_dl:
            out = model(x.to(device))
            preds += out.argmax(1).cpu().tolist()
            labels += y.tolist()
    acc = np.mean(np.array(preds) == np.array(labels))
    f1 = f1_score(labels, preds, average="macro")
    print(f"Epoch {epoch+1}/{EPOCHS} | loss {total_loss/len(train_dl):.3f} | val acc {acc:.4f} | macro-F1 {f1:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save({"model": model.state_dict(), "classes": classes}, SAVE)
        print("  saved best model")

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_908/3094810904.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/5 | loss 0.826 | val acc 0.9559 | macro-F1 0.9411
  saved best model
Epoch 2/5 | loss 0.162 | val acc 0.9770 | macro-F1 0.9736
  saved best model
Epoch 3/5 | loss 0.063 | val acc 0.9918 | macro-F1 0.9913
  saved best model
Epoch 4/5 | loss 0.018 | val acc 0.9976 | macro-F1 0.9977
  saved best model
Epoch 5/5 | loss 0.007 | val acc 0.9961 | macro-F1 0.9965


In [ ]:
PD_TEST = "/content/data/plantdoc/test"
pd_classes = sorted(os.listdir(PD_TEST))
for c in pd_classes:
    print(f"{c}: {len(os.listdir(f'{PD_TEST}/{c}'))} images")

print("\nOur model's classes:")
for c in classes:
    print(" ", c)

Apple Scab Leaf: 10 images
Apple leaf: 9 images
Apple rust leaf: 10 images
Bell_pepper leaf: 8 images
Bell_pepper leaf spot: 9 images
Blueberry leaf: 11 images
Cherry leaf: 10 images
Corn Gray leaf spot: 4 images
Corn leaf blight: 12 images
Corn rust leaf: 10 images
Peach leaf: 9 images
Potato leaf early blight: 8 images
Potato leaf late blight: 8 images
Raspberry leaf: 7 images
Soyabean leaf: 8 images
Squash Powdery mildew leaf: 6 images
Strawberry leaf: 8 images
Tomato Early blight leaf: 9 images
Tomato Septoria leaf spot: 11 images
Tomato leaf: 8 images
Tomato leaf bacterial spot: 9 images
Tomato leaf late blight: 10 images
Tomato leaf mosaic virus: 10 images
Tomato leaf yellow virus: 6 images
Tomato mold leaf: 6 images
grape leaf: 12 images
grape leaf black rot: 8 images

Our model's classes:
  Pepper__bell___Bacterial_spot
  Pepper__bell___healthy
  Potato___Early_blight
  Potato___Late_blight
  Potato___healthy
  Tomato_Bacterial_spot
  Tomato_Early_blight
  Tomato_Late_blight
  

In [ ]:
from PIL import Image
from sklearn.metrics import confusion_matrix

PD_TEST = "/content/data/plantdoc/test"

# PlantDoc folder name -> our class name
mapping = {
    "Bell_pepper leaf spot":      "Pepper__bell___Bacterial_spot",
    "Bell_pepper leaf":           "Pepper__bell___healthy",
    "Potato leaf early blight":   "Potato___Early_blight",
    "Potato leaf late blight":    "Potato___Late_blight",
    "Tomato leaf bacterial spot": "Tomato_Bacterial_spot",
    "Tomato Early blight leaf":   "Tomato_Early_blight",
    "Tomato leaf late blight":    "Tomato_Late_blight",
    "Tomato mold leaf":           "Tomato_Leaf_Mold",
    "Tomato Septoria leaf spot":  "Tomato_Septoria_leaf_spot",
    "Tomato leaf yellow virus":   "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato leaf mosaic virus":   "Tomato__Tomato_mosaic_virus",
    "Tomato leaf":                "Tomato_healthy",
}
cls_idx = {c: i for i, c in enumerate(classes)}

# reload the best checkpoint (epoch 4)
ckpt = torch.load("/content/drive/MyDrive/leaflens/effnet_b0_pv.pt", map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

y_true, y_pred = [], []
per_class = {}
with torch.no_grad():
    for pd_name, our_name in mapping.items():
        folder = f"{PD_TEST}/{pd_name}"
        correct = total = 0
        for f in os.listdir(folder):
            try:
                img = Image.open(f"{folder}/{f}").convert("RGB")
            except Exception:
                continue
            x = val_tf(img).unsqueeze(0).to(device)
            pred = model(x).argmax(1).item()
            y_true.append(cls_idx[our_name]); y_pred.append(pred)
            total += 1
            correct += int(pred == cls_idx[our_name])
        per_class[pd_name] = (correct, total)

for k, (c, t) in per_class.items():
    print(f"{k:32s} {c}/{t}  ({100*c/t:.0f}%)")

acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\nPlantDoc accuracy (12 overlapping classes): {acc:.3f}  on {len(y_true)} images")
print(f"PlantVillage val accuracy was: {best_acc:.3f}")

Bell_pepper leaf spot            2/9  (22%)
Bell_pepper leaf                 3/8  (38%)
Potato leaf early blight         4/8  (50%)
Potato leaf late blight          0/8  (0%)
Tomato leaf bacterial spot       0/9  (0%)
Tomato Early blight leaf         4/9  (44%)
Tomato leaf late blight          6/10  (60%)
Tomato mold leaf                 0/6  (0%)
Tomato Septoria leaf spot        4/11  (36%)
Tomato leaf yellow virus         0/6  (0%)
Tomato leaf mosaic virus         0/10  (0%)
Tomato leaf                      0/8  (0%)

PlantDoc accuracy (12 overlapping classes): 0.225  on 102 images
PlantVillage val accuracy was: 0.998


In [ ]:
from collections import Counter

for pd_name in ["Tomato leaf", "Tomato leaf mosaic virus", "Tomato leaf bacterial spot"]:
    folder = f"{PD_TEST}/{pd_name}"
    preds = []
    with torch.no_grad():
        for f in os.listdir(folder):
            try:
                img = Image.open(f"{folder}/{f}").convert("RGB")
            except Exception:
                continue
            preds.append(classes[model(val_tf(img).unsqueeze(0).to(device)).argmax(1).item()])
    print(f"\n{pd_name} -> predicted as:")
    for k, v in Counter(preds).most_common(4):
        print(f"   {k}: {v}")


Tomato leaf -> predicted as:
   Tomato_Late_blight: 3
   Tomato_Early_blight: 3
   Pepper__bell___healthy: 2

Tomato leaf mosaic virus -> predicted as:
   Tomato_Early_blight: 5
   Tomato_Septoria_leaf_spot: 2
   Tomato_Late_blight: 1
   Pepper__bell___healthy: 1

Tomato leaf bacterial spot -> predicted as:
   Tomato_Septoria_leaf_spot: 3
   Tomato_Late_blight: 2
   Tomato_Early_blight: 2
   Pepper__bell___healthy: 1


In [ ]:
import torch.nn as nn
from torch.utils.data import ConcatDataset

PD_TRAIN = "/content/data/plantdoc/train"

# --- strong "field-like" augmentation ---
strong_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.3, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomPerspective(0.3, p=0.4),
    transforms.ColorJitter(0.5, 0.5, 0.5, 0.1),
    transforms.RandomGrayscale(0.05),
    transforms.GaussianBlur(5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.2)),
])

# PlantVillage train split with strong aug
pv_strong = torch.utils.data.Subset(datasets.ImageFolder(PV, transform=strong_tf), train_idx)

# PlantDoc train (only the 12 classes we can map)
class PDDataset(torch.utils.data.Dataset):
    def __init__(self, root, mapping, tf):
        self.items, self.tf = [], tf
        for pd_name, our_name in mapping.items():
            folder = f"{root}/{pd_name}"
            if not os.path.isdir(folder):
                print("  missing in train:", pd_name); continue
            for f in os.listdir(folder):
                self.items.append((f"{folder}/{f}", cls_idx[our_name]))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        try:
            img = Image.open(p).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224))
        return self.tf(img), y

pd_train = PDDataset(PD_TRAIN, mapping, strong_tf)
print("PlantDoc train images used:", len(pd_train))

# oversample PlantDoc x5 since it's small
combined = ConcatDataset([pv_strong] + [pd_train] * 5)
comb_dl = DataLoader(combined, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
print("Combined training set:", len(combined))

# preload PlantDoc test once so eval is fast
test_items = []
for pd_name, our_name in mapping.items():
    for f in os.listdir(f"{PD_TEST}/{pd_name}"):
        try:
            img = Image.open(f"{PD_TEST}/{pd_name}/{f}").convert("RGB")
            test_items.append((val_tf(img), cls_idx[our_name]))
        except Exception:
            pass

def eval_pd():
    model.eval(); ok = 0
    with torch.no_grad():
        for x, y in test_items:
            ok += int(model(x.unsqueeze(0).to(device)).argmax(1).item() == y)
    return ok / len(test_items)

# --- fine-tune from the epoch-4 checkpoint ---
ckpt = torch.load("/content/drive/MyDrive/leaflens/effnet_b0_pv.pt", map_location=device)
model.load_state_dict(ckpt["model"])

EPOCHS = 6
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, steps_per_epoch=len(comb_dl), epochs=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in comb_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item()

    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for x, y in val_dl:
            preds += model(x.to(device)).argmax(1).cpu().tolist(); labels += y.tolist()
    pv_acc = np.mean(np.array(preds) == np.array(labels))
    print(f"Epoch {epoch+1}/{EPOCHS} | loss {total/len(comb_dl):.3f} | PV val {pv_acc:.4f} | PlantDoc test {eval_pd():.3f}")

# save the FINAL epoch (not the best-on-test one, to keep the test set honest)
torch.save({"model": model.state_dict(), "classes": classes},
           "/content/drive/MyDrive/leaflens/effnet_b0_field.pt")
print("saved")

PlantDoc train images used: 996
Combined training set: 21490


/tmp/ipykernel_908/892778762.py:87: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


Epoch 1/6 | loss 1.951 | PV val 0.9634 | PlantDoc test 0.314
Epoch 2/6 | loss 1.090 | PV val 0.9874 | PlantDoc test 0.441
Epoch 3/6 | loss 0.936 | PV val 0.9852 | PlantDoc test 0.510
Epoch 4/6 | loss 0.840 | PV val 0.9906 | PlantDoc test 0.578
Epoch 5/6 | loss 0.779 | PV val 0.9930 | PlantDoc test 0.569
Epoch 6/6 | loss 0.751 | PV val 0.9915 | PlantDoc test 0.549
saved


In [ ]:
from collections import Counter

model.eval()
print("Per-class accuracy after fine-tuning:\n")
for pd_name, our_name in mapping.items():
    folder = f"{PD_TEST}/{pd_name}"
    preds = []
    with torch.no_grad():
        for f in os.listdir(folder):
            try:
                img = Image.open(f"{folder}/{f}").convert("RGB")
            except Exception:
                continue
            preds.append(classes[model(val_tf(img).unsqueeze(0).to(device)).argmax(1).item()])
    c = sum(p == our_name for p in preds)
    print(f"{pd_name:32s} {c}/{len(preds)}  top wrong: {[k for k,_ in Counter(p for p in preds if p != our_name).most_common(2)]}")

Per-class accuracy after fine-tuning:

Bell_pepper leaf spot            6/9  top wrong: ['Potato___Early_blight', 'Tomato_Bacterial_spot']
Bell_pepper leaf                 8/8  top wrong: []
Potato leaf early blight         3/8  top wrong: ['Potato___Late_blight', 'Tomato_Early_blight']
Potato leaf late blight          5/8  top wrong: ['Potato___Early_blight']
Tomato leaf bacterial spot       2/9  top wrong: ['Tomato_Septoria_leaf_spot', 'Pepper__bell___Bacterial_spot']
Tomato Early blight leaf         4/9  top wrong: ['Tomato_Leaf_Mold', 'Potato___Early_blight']
Tomato leaf late blight          5/10  top wrong: ['Potato___Late_blight', 'Tomato_Leaf_Mold']
Tomato mold leaf                 3/6  top wrong: ['Tomato_Septoria_leaf_spot', 'Tomato_Late_blight']
Tomato Septoria leaf spot        9/11  top wrong: ['Tomato_Leaf_Mold', 'Tomato__Tomato_YellowLeaf__Curl_Virus']
Tomato leaf yellow virus         5/6  top wrong: ['Pepper__bell___healthy']
Tomato leaf mosaic virus         3/10  top wro

In [ ]:
import torch.nn.functional as F

model.eval()
rows = []  # (max_prob, energy, correct)
with torch.no_grad():
    for x, y in test_items:
        logits = model(x.unsqueeze(0).to(device))
        p = F.softmax(logits, 1)
        rows.append((p.max().item(),
                     -torch.logsumexp(logits, 1).item(),   # energy: lower = more in-distribution
                     int(p.argmax(1).item() == y)))
rows = np.array(rows)

print("Avg confidence when RIGHT:", rows[rows[:,2]==1,0].mean().round(3))
print("Avg confidence when WRONG:", rows[rows[:,2]==0,0].mean().round(3))

print("\nthreshold | answered | accuracy on answered")
for t in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = rows[:,0] >= t
    if keep.sum():
        print(f"  {t:.1f}    | {keep.mean():.0%}     | {rows[keep,2].mean():.0%}")

# OOD check: PlantDoc crops our model never saw (apple, corn, grape...)
unseen = [c for c in os.listdir(PD_TEST) if c not in mapping]
ood_max = []
with torch.no_grad():
    for c in unseen:
        for f in os.listdir(f"{PD_TEST}/{c}"):
            try:
                img = Image.open(f"{PD_TEST}/{c}/{f}").convert("RGB")
            except Exception:
                continue
            ood_max.append(F.softmax(model(val_tf(img).unsqueeze(0).to(device)), 1).max().item())
print(f"\nUnseen-crop images: {len(ood_max)}, avg confidence: {np.mean(ood_max):.3f}")

Avg confidence when RIGHT: 0.657
Avg confidence when WRONG: 0.541

threshold | answered | accuracy on answered
  0.4    | 81%     | 59%
  0.5    | 66%     | 66%
  0.6    | 52%     | 66%
  0.7    | 34%     | 63%
  0.8    | 24%     | 62%
  0.9    | 8%     | 88%

Unseen-crop images: 134, avg confidence: 0.530


In [ ]:
from sklearn.metrics import roc_auc_score

def get_logits(img):
    with torch.no_grad():
        return model(val_tf(img).unsqueeze(0).to(device))[0].cpu()

# in-distribution logits (PlantDoc test, mapped classes)
in_logits, in_y = [], []
for x, y in test_items:
    with torch.no_grad():
        in_logits.append(model(x.unsqueeze(0).to(device))[0].cpu()); in_y.append(y)
in_logits = torch.stack(in_logits); in_y = np.array(in_y)

# unseen-crop logits
out_logits = []
for c in unseen:
    for f in os.listdir(f"{PD_TEST}/{c}"):
        try: out_logits.append(get_logits(Image.open(f"{PD_TEST}/{c}/{f}").convert("RGB")))
        except Exception: pass
out_logits = torch.stack(out_logits)

def scores(l):
    return {"max softmax": F.softmax(l,1).max(1).values.numpy(),
            "max logit":   l.max(1).values.numpy(),
            "energy (-E)": torch.logsumexp(l,1).numpy()}

si, so = scores(in_logits), scores(out_logits)
print("OOD AUROC (unseen crops vs target crops), 0.5 = useless, 1.0 = perfect:")
for k in si:
    auc = roc_auc_score(np.r_[np.ones(len(si[k])), np.zeros(len(so[k]))], np.r_[si[k], so[k]])
    print(f"  {k:13s} {auc:.3f}")

# crop-restricted accuracy
crop_of = np.array([c.split("_")[0] for c in classes])
masked_ok = 0
for l, y in zip(in_logits, in_y):
    m = torch.tensor(crop_of != crop_of[y])
    masked_ok += int(l.masked_fill(m, -1e9).argmax().item() == y)
print(f"\nUnrestricted accuracy:     {np.mean(in_logits.argmax(1).numpy()==in_y):.3f}")
print(f"Crop-restricted accuracy:  {masked_ok/len(in_y):.3f}")

OOD AUROC (unseen crops vs target crops), 0.5 = useless, 1.0 = perfect:
  max softmax   0.598
  max logit     0.638
  energy (-E)   0.646

Unrestricted accuracy:     0.549
Crop-restricted accuracy:  0.647


In [ ]:
# top-1 / top-2 / top-3, with crop restriction
ranks = []
for l, y in zip(in_logits, in_y):
    m = torch.tensor(crop_of != crop_of[y])
    order = l.masked_fill(m, -1e9).argsort(descending=True).tolist()
    ranks.append(order.index(y))
ranks = np.array(ranks)
for k in [1, 2, 3]:
    print(f"Crop-restricted top-{k} accuracy: {(ranks < k).mean():.3f}")

# energy threshold: what share of photos get a "not very sure" warning
e_in = -torch.logsumexp(in_logits, 1).numpy()       # lower = more confident
thr = np.percentile(e_in, 75)                       # warn on the least-confident 25%
warn = e_in > thr
correct = (in_logits.argmax(1).numpy() == in_y)
print(f"\nWarned on 25% of photos. Accuracy on NOT-warned: {correct[~warn].mean():.3f}")
print(f"Accuracy on warned:                  {correct[warn].mean():.3f}")

Crop-restricted top-1 accuracy: 0.647
Crop-restricted top-2 accuracy: 0.824
Crop-restricted top-3 accuracy: 0.902

Warned on 25% of photos. Accuracy on NOT-warned: 0.579
Accuracy on warned:                  0.462


In [ ]:
!pip install -q onnx onnxscript onnxruntime

In [ ]:
import json

# save threshold + class list next to the model
meta = {"classes": classes, "energy_warn_threshold": float(thr)}
json.dump(meta, open("/content/drive/MyDrive/leaflens/model_meta.json", "w"), indent=2)

# export to ONNX
model.eval()
dummy = torch.randn(1, 3, 224, 224).to(device)
onnx_path = "/content/drive/MyDrive/leaflens/leaflens.onnx"
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

import onnxruntime as ort  # pip install onnxruntime if missing
sess = ort.InferenceSession(onnx_path)
out = sess.run(None, {"input": dummy.cpu().numpy()})[0]
print("ONNX output shape:", out.shape)
print("Max diff vs PyTorch:", np.abs(out - model(dummy).detach().cpu().numpy()).max())
print("Size (MB):", round(os.path.getsize(onnx_path) / 1e6, 1))

/tmp/ipykernel_908/1455996164.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1103 14:21:39.072000 908 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:55: 

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX output shape: (1, 15)
Max diff vs PyTorch: 8.821487e-06
Size (MB): 0.5


In [ ]:
!ls -lh /content/drive/MyDrive/leaflens/

total 48M
drwx------ 3 root root 4.0K Nov  3 12:47 datasets
-rw------- 1 root root  16M Nov  3 13:58 effnet_b0_field.pt
-rw------- 1 root root  16M Nov  3 13:13 effnet_b0_pv.pt
-rw------- 1 root root 504K Nov  3 14:22 leaflens.onnx
-rw------- 1 root root  16M Nov  3 14:22 leaflens.onnx.data
-rw------- 1 root root  536 Nov  3 14:21 model_meta.json


In [ ]:
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=18,
    dynamo=False,
)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
out = sess.run(None, {"input": dummy.cpu().numpy()})[0]
print("Output shape:", out.shape)
print("Max diff:", np.abs(out - model(dummy).detach().cpu().numpy()).max())
print("Size (MB):", round(os.path.getsize(onnx_path) / 1e6, 1))

/tmp/ipykernel_908/2716868057.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Output shape: (1, 15)
Max diff: 9.000301e-06
Size (MB): 16.1


In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/leaflens/model_meta.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>